# TD: RAG

Dans ce notebook, un RAG basique est implémenté:
- On chunk les documents par paragraphes
- On a un embedding pour les chunks
- Pour une question, on peut embedde la question et récupérer les N chunks les plus pertinents
- On utilise un modèle de génération de texte (SMoLL) pour faire la partie question + chunks les plus pertinents -> réponse.

Téléchargez (cette archive)[https://drive.google.com/file/d/1TnfKs7bTwmpbXklbgiIBpdw7I_wJ5y9Y/view?usp=sharing] avec différentes

Dans ce TD, vous allez expérimenter différentes façons de chunk et d'embeded les documents et les questions pour que le RAG retrieve les documents les plus pertinents. <br/>
Vous expérimenterez aussi la prompt donnée au générateur de texte pour avoir les meilleures réponses.

Voici la [liste de questions](https://drive.google.com/file/d/14hZ0hTx5dM1WgJYewZsn9BkHzEReq-pj/view?usp=sharing) que je poserai au RAG. </br>
A rendre:
- Le notebook de votre RAG
- un CSV avec question,embedding,rag_reply
- un CSV avec chunk,embedding</br>
L'embedding doit être le JSON d'une liste de float.</br>
Quand je ferai "json.loads(embedding)", je dois récupérer une liste de floats

In [1]:
!pip install transformers==4.56.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.6.0
    Uninstalling huggingface_hub-1.6.0:
      Successfully uninstalled huggingface_hub-1.6.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
!pip install flagembedding

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.3 MB/s eta 0:00:00
  Created wheel for flagembedding: filename=FlagEmbedding-1.3.5-py3-none-any.whl size=233746 sha256=bcfe8d2b77d8efc38fdb623bd07c82625cfbaa31c5f248419164767f578b9933
  Stored in directory: /root/.cache/pip/wheels/b2/1f/f6/78f862bb80cb959cc9960b7c4e2d1f702b1bc0e79d19b5f124
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=710ca840075fb72f8eefa6eebbab4d9c253d22f84a4f964e5c28581370c798cc
  Stored in directory: /root/.c

In [3]:
from FlagEmbedding import FlagModel
# If there is an error, it's because of transformers 5.0+, that's maybe pre-installed on Colab
# The pip install was there to fix it
# Please do Runtime > Restart session
# It should fix it

In [4]:
import numpy as np

import pandas as pd
from pathlib import Path

# Data loading

In [ ]:
path = Path("../data/raw/rag/")

In [ ]:
texts = []
for filename in path.glob("*.md"):
    with open(filename) as f:
        texts.append(f.read())

texts[0]

'# Title: Introduction to Cybersecurity: Principles and Practices  \n\n**Teacher:** Professor Lydia Carter  \n\n**Description:**  \nThis course introduces the fundamentals of cybersecurity, focusing on protecting systems, networks, and data from cyber threats. Students will explore key topics such as cryptography, network security, ethical hacking, and risk management. Through practical labs and real-world case studies, students will gain hands-on experience in identifying vulnerabilities, implementing security measures, and understanding the legal and ethical aspects of cybersecurity.  \n\n**Prerequisites:**  \n- Basic knowledge of computer networks and operating systems  \n- Proficiency in at least one programming language (e.g., Python, Java, or C++)  \n- Completion of "Introduction to Computer Science" or equivalent  \n\n**Assessment:**  \n- Weekly quizzes and assignments (25%)  \n- Midterm exam: Fundamentals of cybersecurity (20%)  \n- Final project: Design and present a comprehen

# Chunk
## Basic

In [ ]:
def parse_class(text):
    chunks = text.split("\n\n")
    title = chunks[0].replace("# Title: ", "")
    return {"title": title, "chunks": chunks}

In [ ]:
def parse_class_add_title(text):
    chunks = text.split("\n\n")
    title = chunks[0].replace("# Title: ", "")
    return {"title": title, "chunks": [f"{title}: {chunk}" for chunk in chunks]}

In [ ]:
chunks = sum((parse_class_add_title(txt)["chunks"] for txt in texts), [])

# Embedding

## BAAI's embedding

In [ ]:
model = FlagModel(
    'BAAI/bge-base-en-v1.5',
    query_instruction_for_retrieval="Represent this sentence for searching relevant passages:",
    use_fp16=True,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [ ]:
corpus_embedding = model.encode(chunks)

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [ ]:
queries = [
    "Who is the reinforcement learning teacher?",
    "In what class will I learn game AI?",
]

In [ ]:
query_embedding = model.encode(queries)

In [ ]:
sim_scores = query_embedding @ corpus_embedding.T

In [ ]:
for query, score in zip(queries, sim_scores):
    print(" ---- ")
    print("Query: ", query)
    indexes = np.argsort(score)[-5:]
    print("Sources:")
    for i, idx in enumerate(reversed(indexes)):
        if score[idx] > .5:
            print(f"{i+1} -- similarity {score[idx]:.2f} -- \"", chunks[idx], '"')



 ---- 
Query:  Who is the reinforcement learning teacher?
Sources:
1 -- similarity 0.80 -- " Foundations of Reinforcement Learning  : **Teacher:** Dr. Arjun Patel   "
2 -- similarity 0.74 -- " Foundations of Reinforcement Learning  : # Title: Foundations of Reinforcement Learning   "
3 -- similarity 0.71 -- " Foundations of Reinforcement Learning  : 2. **Tabular Methods**  
   - Dynamic programming approaches: Policy Iteration and Value Iteration  
   - Monte Carlo methods and Temporal-Difference (TD) Learning   "
4 -- similarity 0.71 -- " Foundations of Reinforcement Learning  : 4. **Policy-Based Methods**  
   - Policy Gradient methods and REINFORCE algorithm  
   - Advantage Actor-Critic (A2C) and Proximal Policy Optimization (PPO)   "
5 -- similarity 0.71 -- " Foundations of Reinforcement Learning  : **Description:**  
This course explores the foundational principles and practical applications of reinforcement learning (RL), a branch of machine learning focused on decision-making a

# Eval retrieval: Mean Reciprocal Rank
Le fichier [question_answer_short.csv](https://drive.google.com/file/d/1EB8IwGlqvpNy3oq7xyR2IzdqJDX8C_fr/view?usp=drive_link) contient une liste de question et le texte à retrouver dans les documents.<br/>
Je considère que tout chunk contenant le "texte à retrouver" était un bon chunk

In [ ]:
df = pd.read_csv(path / "question_answer_short.csv")

In [ ]:
query_embedding = model.encode(list(df["question"]))

In [ ]:
acceptable_chunks = []
for answer in df["answer"]:
    chunks_ok = set(i for i, chunk in enumerate(chunks) if answer in chunk)
    acceptable_chunks.append(chunks_ok)

In [ ]:
def compute_mrr(sim_score, acceptable_chunks):
    ranks = []
    for this_score, this_acceptable_chunks in zip(sim_score, acceptable_chunks):
        indexes = reversed(np.argsort(this_score))
        rank = 1 + next(i for i, idx in enumerate(indexes) if idx in this_acceptable_chunks)
        ranks.append(rank)

    return {
        "score": sum(1 / r if r < 6 else 0 for r in ranks) / len(ranks),
        "ranks": ranks,
    }

In [ ]:
sim_scores = query_embedding @ corpus_embedding.T

In [ ]:
res = compute_mrr(sim_scores, acceptable_chunks)
res["score"]

0.6

# Text generation

In [ ]:
def get_context(query, corpus, corpus_embeddings):
    query_embedding = model.encode([query])
    sim_scores = query_embedding @ corpus_embedding.T
    indexes = list(np.argsort(sim_scores[0]))[-5:]
    return [corpus[i] for i in indexes]

In [ ]:
get_context("Which class will teach me to build a chatbot?", chunks, corpus_embedding)

['# Natural Language Processing (NLP) Fundamentals and Applications: 5. **Applications of NLP**\n  - Sentiment analysis and text classification\n  - Machine translation and summarization\n  - Chatbots and conversational agents',
 '# Natural Language Processing (NLP) Fundamentals and Applications: **Description:**\nThis course offers a comprehensive introduction to the field of Natural Language Processing (NLP), focusing on the computational techniques that allow machines to understand, interpret, and generate human language. You will learn about linguistic structures, text preprocessing, sentiment analysis, machine translation, and language modeling. Using hands-on projects and industry-relevant tools, this course provides a strong foundation in both traditional and modern NLP methods, including neural networks and transformers.',
 '# Natural Language Processing (NLP) Fundamentals and Applications: Whether you aim to pursue a career in AI or enhance your programming toolkit, this cours

# Groq generator

In [ ]:
groq_api_key = "YOUR-API-KEY"

In [ ]:
import openai

In [ ]:
client = openai.OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
query = "What must I do to pass the NLP class?"

context_str = "\n\n".join(get_context(query, chunks, corpus_embedding))

prompt = f"""Context information is below.
---------------------
{context_str}
---------------------
Given the context information and not prior knowledge, answer the query.
If the answer is not in the context information, reply "I cannot answer that question".
Query: {query}
Answer:"""

In [ ]:
res = client.chat.completions.create(
    messages=[{"role": "user", "content": prompt}],
    model="openai/gpt-oss-20b",
)

In [ ]:
res.choices[0].message.content

'To pass the NLP class you need to:\n\n1. **Meet the prerequisites**  \n   * Be proficient in Python programming.  \n   * Have a basic understanding of linear algebra and probability.  \n   * Have successfully completed “Introduction to Machine Learning” (or an equivalent course).\n\n2. **Earn the required assessment scores** – the course is weighted as follows:  \n   * **Weekly coding assignments** – 30\u202f% of your grade. Complete every assignment on time and ensure the code runs correctly.  \n   * **Midterm exam** – 20\u202f% of your grade. Study the material covered up to the mid‑term and perform well on the exam.  \n   * **Final project** – 30\u202f% of your grade. Design, implement, and present an end‑to‑end NLP application (e.g., sentiment analyzer, chatbot, or translator). Submit a written report and give a class presentation.  \n   * **Participation** – 20\u202f% of your grade. Actively engage in class discussions, code reviews, and any collaborative activities.\n\n3. **Atte

In [ ]:
from pprint import pprint

In [ ]:
pprint(res.choices[0].message.content)

('To pass the NLP class you need to:\n'
 '\n'
 '1. **Meet the prerequisites**  \n'
 '   * Be proficient in Python programming.  \n'
 '   * Have a basic understanding of linear algebra and probability.  \n'
 '   * Have successfully completed “Introduction to Machine Learning” (or an '
 'equivalent course).\n'
 '\n'
 '2. **Earn the required assessment scores** – the course is weighted as '
 'follows:  \n'
 '   * **Weekly coding assignments** – 30\u202f% of your grade. Complete every '
 'assignment on time and ensure the code runs correctly.  \n'
 '   * **Midterm exam** – 20\u202f% of your grade. Study the material covered '
 'up to the mid‑term and perform well on the exam.  \n'
 '   * **Final project** – 30\u202f% of your grade. Design, implement, and '
 'present an end‑to‑end NLP application (e.g., sentiment analyzer, chatbot, or '
 'translator). Submit a written report and give a class presentation.  \n'
 '   * **Participation** – 20\u202f% of your grade. Actively engage in class '
 '